# 05 --  Model Comparison

## Concept
Compare all baseline + advanced models on multiple metrics: accuracy, precision, recall, F1, ROC AUC, and training time.

## Mathematical Intuition
- **Precision**: TP / (TP + FP) --  how many positive predictions are correct
- **Recall**: TP / (TP + FN) --  how many actual positives are found
- **F1**: 2 x precision x recall / (precision + recall) --  harmonic mean
- **ROC AUC**: probability that model ranks a random positive higher than a random negative

## Interview Questions
1. Why is accuracy not always a good metric?
2. When would you prioritize recall over precision?
3. What does ROC AUC = 0.5 mean? What about ROC AUC = 1.0?

## Production Mapping
Classification metrics are computed by `evaluation/classification.py`. The EvaluationReporter in `evaluation/reporter.py` generates formatted reports.


In [ ]:
import pandas as pd, numpy as np, time, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
np.random.seed(42)
n = 1000
X = pd.DataFrame({
    'capacity_mw': np.random.exponential(500, n),
    'region_risk': np.random.uniform(0, 1, n),
    'age_years': np.random.exponential(30, n),
    'num_connections': np.random.poisson(5, n),
})
y = (X['capacity_mw'] / 100 + X['region_risk'] * 5 + np.random.normal(0, 0.5, n)).clip(0, 3).round().astype(int).clip(0, 3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
}

In [ ]:
results = []
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_s, y_train)
    fit_time = time.time() - t0
    y_pred = model.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5, n_jobs=-1)
    results.append({'model': name, 'accuracy': acc, 'precision': p, 'recall': r, 'f1': f,
                    'fit_time': fit_time, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()})
results_df = pd.DataFrame(results).round(4)
print(results_df.to_string(index=False))

In [ ]:
# ROC AUC (OVR for multiclass)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results_df.plot(x='model', y='accuracy', kind='bar', ax=axes[0], legend=False)
axes[0].set_title('Accuracy by Model')
axes[0].tick_params(axis='x', rotation=45)
results_df.plot(x='model', y='fit_time', kind='bar', ax=axes[1], legend=False, color='orange')
axes[1].set_title('Training Time (seconds)')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../artifacts/05_model_comparison.png', dpi=100)
plt.show()

## Key Takeaways
- XGBoost and LightGBM usually outperform Random Forest
- Training time varies dramatically: LogReg < 0.1s, LightGBM/XGBoost < 1s, RF ~ seconds
- Cross-validation scores reveal stability (low std = stable)
- Use multiple metrics --  accuracy alone is misleading for imbalanced data
- In production, the best model is archived and transitioned through lifecycle stages